In [3]:
print("Nuevo directorio:", os.getcwd())
# ============================================================
# RECODIFICACIÓN DE GRUPO_EDAD A QUINQUENIOS ESTÁNDAR
# Supuesto: décadas se dividen en mitades iguales (n/2)
# Estándar OPS para datos agregados sin microdatos de edad
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

CODIGOS_C16 = ['C160', 'C161', 'C162', 'C163', 'C164', 'C165', 'C166', 'C168', 'C169']
AÑOS = [2018, 2019, 2020, 2021, 2022]

# --- Carga ---
def cargar_csv(ruta):
    try:
        return pd.read_csv(ruta, sep=';', encoding='latin1', low_memory=False)
    except Exception as e:
        print(f"Error al cargar {ruta}: {e}")
        return None

ruta_datos = Path('.')
archivos_csv = list(ruta_datos.glob('*.csv'))

dfs = {}
for archivo in archivos_csv:
    nombre = archivo.name.upper()
    for a in AÑOS:
        if str(a) in nombre:
            df = cargar_csv(archivo)
            if df is not None:
                df['ANO_EGRESO'] = a
                dfs[a] = df
                print(f"Cargado: {archivo.name} - {len(df)} registros")
            break

df_total = pd.concat(dfs.values(), ignore_index=True)
df_c16 = df_total[df_total['DIAG1'].isin(CODIGOS_C16)].copy()
df_c16['SUBTIPO'] = 'TOTAL_C16'
print(f"\nRegistros C16: {len(df_c16)}")
print(f"Por año:\n{df_c16['ANO_EGRESO'].value_counts().sort_index()}")

# --- Limpieza general ---
for col in ['SEXO', 'GRUPO_EDAD', 'GLOSA_REGION_RESIDENCIA', 'CONDICION_EGRESO']:
    if col in df_c16.columns:
        df_c16[col] = df_c16[col].replace('*', 'No reportado').fillna('No reportado')

df_c16['DIAS_ESTADA'] = pd.to_numeric(df_c16['DIAS_ESTADA'], errors='coerce')
df_c16['SEXO'] = df_c16['SEXO'].astype(str).str.upper().str.strip()
df_c16['SEXO'] = df_c16['SEXO'].replace({'1': 'HOMBRE', '2': 'MUJER'})

# ============================================================
# MAPEO DE GRUPO_EDAD A QUINQUENIOS
# Sistema nuevo (quinquenios): mapeo directo
# Sistema antiguo (décadas): división n/2 por quinquenio
# Pediátricos: agrupados en '00-04 años'
# ============================================================

# Orden canónico de quinquenios para output final
QUINQUENIOS_ORDEN = [
    '00-04 años',
    '05-09 años',
    '10-14 años',
    '15-19 años',
    '20-24 años',
    '25-29 años',
    '30-34 años',
    '35-39 años',
    '40-44 años',
    '45-49 años',
    '50-54 años',
    '55-59 años',
    '60-64 años',
    '65-69 años',
    '70-74 años',
    '75-79 años',
    '80-84 años',
    '85 y más años',
    'No reportado',
]

# Mapeo directo: categoría original -> quinquenio(s)
# Valor: quinquenio único, o lista de dos quinquenios si es década (se divide n/2)
MAPA_EDAD = {
    # --- Pediátricos -> 00-04 años ---
    'menor a 7 días':           ['00-04 años'],
    'MENOR DE 7 DIAS':          ['00-04 años'],
    '7 A 27 DÍAS':              ['00-04 años'],
    '28 DIAS A 2 MES':          ['00-04 años'],
    '2 ,ESES A MENOS DE 1 AÑO': ['00-04 años'],  # typo original en datos
    '2 MESES A MENOS DE 1 AÑO': ['00-04 años'],
    'menor de un año':          ['00-04 años'],
    'MENOR DE 1 AÑO':           ['00-04 años'],
    '1 a 4 AÑOS':               ['00-04 años'],
    '1 A 4 AÑOS':               ['00-04 años'],

    # --- Sistema nuevo (quinquenios directos) ---
    '5 A 9 AÑOS':               ['05-09 años'],
    '10 A 14 AÑOS':             ['10-14 años'],
    '15 A 19 AÑOS':             ['15-19 años'],
    '20 A 24 AÑOS':             ['20-24 años'],
    '25 A 29 AÑOS':             ['25-29 años'],
    '30 A 34 AÑOS':             ['30-34 años'],
    '35 A 39 AÑOS':             ['35-39 años'],
    '40 A 44 AÑOS':             ['40-44 años'],
    '45 A 49 AÑOS':             ['45-49 años'],
    '50 A 54 AÑOS':             ['50-54 años'],
    '55 A 59 AÑOS':             ['55-59 años'],
    '60 A 64 AÑOS':             ['60-64 años'],
    '65 A 69 AÑOS':             ['65-69 años'],
    '70 A 74 AÑOS':             ['70-74 años'],
    '75 A 79 AÑOS':             ['75-79 años'],
    '80 A 84 AÑOS':             ['80-84 años'],
    '85 A MAS':                 ['85 y más años'],
    '85 Y MAS':                 ['85 y más años'],

    # --- Sistema antiguo (décadas) -> dos quinquenios, n/2 cada uno ---
    '1 a 9':    ['05-09 años'],        # solo un quinquenio relevante post-infancia
    '1 A 9':    ['05-09 años'],
    '10 a 19':  ['10-14 años', '15-19 años'],
    '10 A 19':  ['10-14 años', '15-19 años'],
    '20 a 29':  ['20-24 años', '25-29 años'],
    '20 A 29':  ['20-24 años', '25-29 años'],
    '30 a 39':  ['30-34 años', '35-39 años'],
    '30 A 39':  ['30-34 años', '35-39 años'],
    '40 a 49':  ['40-44 años', '45-49 años'],
    '40 A 49':  ['40-44 años', '45-49 años'],
    '50 a 59':  ['50-54 años', '55-59 años'],
    '50 A 59':  ['50-54 años', '55-59 años'],
    '60 a 69':  ['60-64 años', '65-69 años'],
    '60 A 69':  ['60-64 años', '65-69 años'],
    '70 a 79':  ['70-74 años', '75-79 años'],
    '70 A 79':  ['70-74 años', '75-79 años'],
    '80 a 89':  ['80-84 años', '85 y más años'],
    '80 A 89':  ['80-84 años', '85 y más años'],
    '90 y más': ['85 y más años'],
    '90 Y MÁS': ['85 y más años'],
    '90 Y MAS': ['85 y más años'],
}

def expandir_por_quinquenio(df):
    """
    Expande registros con décadas en dos filas con peso 0.5 cada una.
    Registros con quinquenio directo mantienen peso 1.0.
    Devuelve df con columnas QUINQUENIO y PESO.
    """
    filas = []
    col_edad = df['GRUPO_EDAD'].astype(str).str.strip()

    for idx, row in df.iterrows():
        edad_raw = col_edad[idx]
        # Buscar en mapa (case-insensitive como respaldo)
        quinquenios = MAPA_EDAD.get(edad_raw)
        if quinquenios is None:
            # Intento case-insensitive
            edad_upper = edad_raw.upper()
            quinquenios = MAPA_EDAD.get(edad_upper, ['No reportado'])

        peso = 1.0 / len(quinquenios)  # 1.0 si directo, 0.5 si década
        for q in quinquenios:
            nueva_fila = row.copy()
            nueva_fila['QUINQUENIO'] = q
            nueva_fila['PESO'] = peso
            filas.append(nueva_fila)

    return pd.DataFrame(filas)

print("\nExpandiendo registros por quinquenio...")
df_expandido = expandir_por_quinquenio(df_c16)
print(f"Registros originales: {len(df_c16)}")
print(f"Registros expandidos: {len(df_expandido)}")
print(f"\nDistribución QUINQUENIO:\n{df_expandido['QUINQUENIO'].value_counts().sort_index()}")

# Verificar categorías no mapeadas
no_mapeados = df_expandido[df_expandido['QUINQUENIO'] == 'No reportado']['GRUPO_EDAD'].value_counts()
if len(no_mapeados) > 0:
    print(f"\n⚠️  Categorías NO mapeadas (quedaron como 'No reportado'):")
    print(no_mapeados)
else:
    print("\n✓ Todas las categorías de edad fueron mapeadas correctamente.")

# ============================================================
# CONSTRUCCIÓN DE TABLA FINAL CON COLUMNAS POR AÑO
# ============================================================

VARIABLES_INTERES = {
    'QUINQUENIO':               QUINQUENIOS_ORDEN,
    'GLOSA_REGION_RESIDENCIA':  None,  # orden alfabético
    'CONDICION_EGRESO':         None,
}

def contar_variable(df_sub, var, orden=None):
    """Cuenta con pesos para QUINQUENIO, conteo simple para el resto."""
    if var == 'QUINQUENIO':
        conteos = df_sub.groupby(var)['PESO'].sum().round(0).astype(int)
    else:
        conteos = df_sub[var].value_counts()

    if orden:
        conteos = conteos.reindex([c for c in orden if c in conteos.index], fill_value=0)
    return conteos

def procesar_segmento(df_seg, sexo_label, año_label):
    """Genera filas para un segmento (sexo x año)."""
    resultados = []
    n_total = len(df_seg)

    # Total general
    resultados.append({
        'SEXO': sexo_label, 'VARIABLE': 'TOTAL_GENERAL',
        'CATEGORIA': 'TOTAL_GENERAL', 'N': n_total, 'ANO': año_label
    })

    # Quinquenio (con pesos)
    conteos_q = contar_variable(df_seg, 'QUINQUENIO', QUINQUENIOS_ORDEN)
    for cat, n in conteos_q.items():
        resultados.append({
            'SEXO': sexo_label, 'VARIABLE': 'GRUPO_EDAD_QUINQUENIO',
            'CATEGORIA': cat, 'N': n, 'ANO': año_label
        })
    resultados.append({
        'SEXO': sexo_label, 'VARIABLE': 'GRUPO_EDAD_QUINQUENIO',
        'CATEGORIA': 'TOTAL_GRUPO_EDAD', 'N': n_total, 'ANO': año_label
    })

    # Región
    for cat, n in df_seg['GLOSA_REGION_RESIDENCIA'].value_counts().items():
        resultados.append({
            'SEXO': sexo_label, 'VARIABLE': 'GLOSA_REGION_RESIDENCIA',
            'CATEGORIA': str(cat), 'N': n, 'ANO': año_label
        })
    resultados.append({
        'SEXO': sexo_label, 'VARIABLE': 'GLOSA_REGION_RESIDENCIA',
        'CATEGORIA': 'TOTAL_REGION', 'N': n_total, 'ANO': año_label
    })

    # Condición de egreso
    for cat, n in df_seg['CONDICION_EGRESO'].value_counts().items():
        resultados.append({
            'SEXO': sexo_label, 'VARIABLE': 'CONDICION_EGRESO',
            'CATEGORIA': str(cat), 'N': n, 'ANO': año_label
        })
    resultados.append({
        'SEXO': sexo_label, 'VARIABLE': 'CONDICION_EGRESO',
        'CATEGORIA': 'TOTAL_CONDICION_EGRESO', 'N': n_total, 'ANO': año_label
    })

    return resultados

# Procesar todos los cruces año x sexo
resultados_totales = []

for año in AÑOS + [None]:
    año_label = str(año) if año else 'TOTAL_5A'

    if año:
        df_año = df_expandido[df_expandido['ANO_EGRESO'] == año]
    else:
        df_año = df_expandido

    for sexo_label in ['GENERAL', 'HOMBRE', 'MUJER']:
        if sexo_label == 'GENERAL':
            df_seg = df_año
        else:
            df_seg = df_año[df_año['SEXO'] == sexo_label]

        resultados_totales += procesar_segmento(df_seg, sexo_label, año_label)

df_resultado = pd.DataFrame(resultados_totales)

# --- Pivotar: años como columnas ---
df_pivot = df_resultado.pivot_table(
    index=['SEXO', 'VARIABLE', 'CATEGORIA'],
    columns='ANO',
    values='N',
    aggfunc='first'
).reset_index()

# Ordenar columnas
cols_año = [str(a) for a in AÑOS] + ['TOTAL_5A']
cols_presentes = [c for c in cols_año if c in df_pivot.columns]
df_pivot = df_pivot[['SEXO', 'VARIABLE', 'CATEGORIA'] + cols_presentes]

# Ordenar filas
orden_sexo  = {'GENERAL': 0, 'HOMBRE': 1, 'MUJER': 2}
orden_var   = {'TOTAL_GENERAL': 0, 'GRUPO_EDAD_QUINQUENIO': 1,
               'GLOSA_REGION_RESIDENCIA': 2, 'CONDICION_EGRESO': 3}
orden_edad  = {q: i for i, q in enumerate(QUINQUENIOS_ORDEN)}

df_pivot['_os'] = df_pivot['SEXO'].map(orden_sexo)
df_pivot['_ov'] = df_pivot['VARIABLE'].map(orden_var)
df_pivot['_oc'] = df_pivot.apply(
    lambda r: orden_edad.get(r['CATEGORIA'], 999) if r['VARIABLE'] == 'GRUPO_EDAD_QUINQUENIO'
    else r['CATEGORIA'], axis=1
)
df_pivot = (df_pivot
    .sort_values(['_os', '_ov', '_oc'])
    .drop(columns=['_os', '_ov', '_oc'])
    .reset_index(drop=True))

print(f"\nTabla final: {df_pivot.shape}")
print(df_pivot.head(60).to_string())

# --- Guardar ---
df_pivot.to_csv('tabla_egreso_cancer_gastrico_quinquenios.csv', index=False, sep=';')
df_pivot.to_excel('tabla_egreso_cancer_gastrico_quinquenios.xlsx', index=False, engine='openpyxl')
print("\nArchivos guardados.")

# ============================================================
# NOTA METODOLÓGICA (para manuscrito)
# ============================================================
print("""
NOTA METODOLÓGICA - GRUPO ETARIO:
Los registros con codificación decenal (sistema anterior al 2021 aprox.)
fueron redistribuidos en quinquenios asumiendo distribución uniforme dentro
de cada década (peso = 0.5 por quinquenio). Los grupos pediátricos menores
de 5 años fueron consolidados en '00-04 años'. Los valores 'No reportado'
se mantienen como categoría explícita. Esta aproximación sigue el estándar
OPS/OMS para armonización de series temporales con cambios en codificación.
""")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Nuevo directorio: /content/drive/MyDrive/INVESTIGACIONES/Descripción epidemiológica del cáncer gástrico en Chile/Datos egresos
Cargado: EGRE_DATOS_ABIERTOS_2018.csv - 1669602 registros
Cargado: EGRE_DATOS_ABIERTOS_2019.csv - 1667180 registros
Cargado: EGRE_DATOS_ABIERTOS_2020.csv - 1330477 registros
Cargado: EGR_DATOS_ABIERTO_2021.csv - 1467062 registros
Cargado: EGRE_DATOS_ABIERTOS_2022.csv - 1597118 registros

Registros C16: 26332
Por año:
ANO_EGRESO
2018    5430
2019    5608
2020    4870
2021    4790
2022    5634
Name: count, dtype: int64

Expandiendo registros por quinquenio...
Registros originales: 26332
Registros expandidos: 47229

Distribución QUINQUENIO:
QUINQUENIO
00-04 años          2
05-09 años          3
10-14 años         16
15-19 años         17
20-24 años        106
25-29 años        126
30-34 años        639
35-39 años        669
40-44 año